In [34]:
import os

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.init as init

import numpy as np

import albumentations as A

import SimpleITK
import SimpleITK as sitk

import random

### 1. General functions


#### 1.1. Configuration


In [35]:
BASE_DIR = "database/"  # Name of the dataset folder

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():  # mac
    device = torch.device("mps")
else:
    device = torch.device("cpu")


print("Using device:", device)

sitk.ProcessObject_SetGlobalWarningDisplay(False)

Using device: mps


#### 1.2. Get files


In [36]:
def get_file_paths(directory):
    """
    Get all file paths in a directory and its subdirectories.
    """
    file_paths = []  # creates an empty list
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_paths.append(os.path.join(root, file))
    return file_paths


all_files = get_file_paths(BASE_DIR)

img_paths = [
    file
    for file in all_files
    if file.endswith(".nii.gz") and "_gt" not in file and "_4d" not in file
]
train_val_paths = [file for file in img_paths if "training" in file]


### 1.3. Transformations for data augmentation


In [37]:
# p: probability of applying the transformations
train_transform = A.Compose(
    [
        A.Resize(
            height=256, width=256, p=1
        ),  # re-sized all the images to 256 × 256 pixels
        A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-10, 10), p=1),
    ]
)

resized = A.Compose([A.Resize(height=256, width=256, p=1)])

### 1.4. Crop function


In [38]:
ROI_SIZE = 128


def crop_center(img, x_center, y_center, size=ROI_SIZE):
    """
    Crop a fixed-size square around a given center.

    Args:
        img: input image (256x256)
        x_center, y_center: center coordinates of ROI
        size: output crop size (default 128x128)

    Returns:
        Cropped image of size x size
    """
    _, height, width = img.shape
    half = size // 2

    x_start = int(np.clip(np.round(x_center - half), 0, width - size))
    y_start = int(np.clip(np.round(y_center - half), 0, height - size))

    return img[:, y_start : y_start + size, x_start : x_start + size]


In [39]:
def calculate_centroid(mask_np):
    """
    Calculates the centroid (y, x) of a binary mask.
    Returns (y_mean, x_mean) or None if the mask is empty.
    """
    y_indices, x_indices = np.where(mask_np > 0)
    if len(y_indices) == 0:
        return None
    return (np.mean(y_indices), np.mean(x_indices))


## 1. FCN2


### 1.1 FCN2 - Dataset


In [40]:
NOISE = 15


class FCN2Dataset(Dataset):
    def __init__(self, filepath, transform=None, mode="train"):
        self.slices = []
        self.transform = transform
        self.mode = mode
        self.noise = NOISE

        for path in filepath:
            img = self._sitk_load(path, normalize=True)

            mask_path = path.replace(".nii.gz", "_gt.nii.gz")
            mask = self._sitk_load(mask_path)

            for i in range(img.shape[0]):
                self.slices.append({"image": img[i], "mask": mask[i]})

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        item = self.slices[idx]
        img = item["image"]
        mask = item["mask"]

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        channel_two = (mask == 2).astype(np.float32)  # myocardium
        channel_three = (mask == 3).astype(np.float32)  # cavity
        new_mask = np.stack([channel_two, channel_three], axis=0)

        new_mask = torch.from_numpy(new_mask).float()
        img = torch.from_numpy(img).float().unsqueeze(0)

        # Testing mode: Return full image
        # FCN1 will handle the cropping dynamically in the evaluation loop.
        if self.mode == "test":
            return img, new_mask

        # Calculate Ground Truth Centroid
        mask_3 = (mask == 3).astype(np.float32)
        gt_centroid = calculate_centroid(mask_3)

        # Training: add noise to the ground-truth centroid and crop the image accordingly
        if gt_centroid is not None:
            y1, x1 = gt_centroid

            if self.mode == "train":
                x0, y0 = (
                    random.randint(-self.noise, self.noise),
                    random.randint(-self.noise, self.noise),
                )  # noise

            elif self.mode == "val":
                x0, y0 = (0, 0)
            noise_centroid = (float(x0 + x1), float(y0 + y1))
        else:
            noise_centroid = (64.0, 64.0)

        crop_img = crop_center(
            img,
            x_center=noise_centroid[0],
            y_center=noise_centroid[1],
            size=ROI_SIZE,
        )
        crop_mask = crop_center(
            new_mask,
            x_center=noise_centroid[0],
            y_center=noise_centroid[1],
            size=ROI_SIZE,
        )
        return crop_img, crop_mask

    def _sitk_load(self, filepath, normalize=False) -> np.ndarray:
        """Loads an image using SimpleITK and returns the image and its metadata.
        Args:
            filepath: Path to the image.

        Returns:
            - ([N], H, W), Image array.

        """
        # Load image and save info
        image = SimpleITK.ReadImage(str(filepath))
        # Extract numpy array from the SimpleITK image object
        im_array = np.squeeze(SimpleITK.GetArrayFromImage(image))
        if normalize:
            im_array = im_array.astype(np.float32)
            im_array = (im_array - im_array.mean()) / (im_array.std() + 1e-8)

        return im_array


In [41]:
# Choose 10 random patients from pattient 001 to patiento 100 for the validation set
random.seed(42)
patients = list(range(1, 101))  # patient001 - patient100
val_patients = random.sample(patients, 10)

train_paths = []
validation_paths = []

# Extracts the patient number from the file path and checks whether that patient belongs to the validation set.
for f in train_val_paths:
    patient = int(f.split("patient")[1][:3])
    if patient in val_patients:
        validation_paths.append(f)
    else:
        train_paths.append(f)

ds_train_fcn2 = FCN2Dataset(train_paths, train_transform, "train")
ds_validation_fcn2 = FCN2Dataset(validation_paths, resized, "val")

### 1.2. FCN1 - Model Architecture


In [42]:
class DoubleEncoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def forward(self, x, skip=None):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        skip = x
        x = self.pool(x)
        return x, skip


class DecoderDouble(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv1 = nn.Conv2d(out_channels * 2, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat((x, skip), dim=1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x


class DecoderSingle(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv1 = nn.Conv2d(out_channels * 2, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat((x, skip), dim=1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        return x


class InceptionModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=1),
            nn.Conv2d(16, 24, kernel_size=3, padding=1),
            nn.Conv2d(24, 24, kernel_size=3, padding=1),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, 12, kernel_size=1),
            nn.Conv2d(12, 16, kernel_size=5, padding=2),
        )
        self.branch3 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, 8, kernel_size=1),
        )
        self.branch4 = nn.Conv2d(in_channels, 16, kernel_size=1)

    def forward(self, x):
        return torch.cat(
            [self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1
        )


class FCN2Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Shared Contracting Path -  Encoder columnxrow
        self.encoder11 = DoubleEncoder(1, 16)
        self.encoder12 = DoubleEncoder(16, 32)
        self.encoder13 = DoubleEncoder(32, 64)
        self.encoder14 = DoubleEncoder(64, 128)

        # Bottleneck 4
        self.bottleneck1 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn_b1 = nn.BatchNorm2d(256)
        self.bottleneck2 = nn.Conv2d(256, 256, 3, padding=1)
        self.bn_b2 = nn.BatchNorm2d(256)
        self.relu = nn.ReLU()

        # --- Expanding Paths (Parallel, independent decoders) ---
        # All paths use DecoderDouble as per the diagram showing 2 convolutions per level

        # Path 4 (Up-sampling from 256 channels)
        self.decode_depth4_step1 = DecoderDouble(256, 128)
        self.decode_depth4_step2 = DecoderDouble(128, 64)
        self.decode_depth4_step3 = DecoderDouble(64, 32)
        self.decode_depth4_step4 = DecoderSingle(32, 16)

        # Path 3 (Up-sampling from 128 channels)
        self.decode_depth3_step1 = DecoderDouble(128, 64)
        self.decode_depth3_step2 = DecoderDouble(64, 32)
        self.decode_depth3_step3 = DecoderSingle(32, 16)

        # Path 2 (Up-sampling from 64 channels)
        self.decode_depth2_step1 = DecoderDouble(64, 32)
        self.decode_depth2_step2 = DecoderSingle(32, 16)

        # Path 1 (Up-sampling from 32 channels)
        self.decode_depth1_step1 = DecoderSingle(32, 16)

        # Inception Module and Final Projection
        # Concatenates 16 channels from each of the 4 paths = 64 input channels
        self.inception = InceptionModule(64)
        # Map to 2 classes (LV cavity and Myocardium)
        self.final_conv = nn.Conv2d(64, 2, 3, padding=1)
        self.sigmoid = nn.Sigmoid()

        # init weights AFTER all layers exist
        self.apply(self.init_weights)

    def forward(self, x):
        # Enconder + skips conections
        x_pool1, s1 = self.encoder11(x)
        x_pool2, s2 = self.encoder12(x_pool1)
        x_pool3, s3 = self.encoder13(x_pool2)
        x_pool4, s4 = self.encoder14(x_pool3)

        # Bottleneck 4
        bottleneck4 = self.bottleneck1(x_pool4)
        bottleneck4 = self.bn_b1(bottleneck4)
        bottleneck4 = self.relu(bottleneck4)
        bottleneck4 = self.bottleneck2(bottleneck4)
        bottleneck4 = self.bn_b2(bottleneck4)
        bottleneck4 = self.relu(bottleneck4)

        # 3. Independent Forward Propagation per Expanding Path
        # No paths are mixed until the final concatenation

        # Path 4
        depth4 = self.decode_depth4_step1(bottleneck4, s4)
        depth4 = self.decode_depth4_step2(depth4, s3)
        depth4 = self.decode_depth4_step3(depth4, s2)
        depth4_out = self.decode_depth4_step4(depth4, s1)

        # Path 3
        depth3 = self.decode_depth3_step1(s4, s3)
        depth3 = self.decode_depth3_step2(depth3, s2)
        depth3_out = self.decode_depth3_step3(depth3, s1)

        # Path 2
        depth2 = self.decode_depth2_step1(s3, s2)
        depth2_out = self.decode_depth2_step2(depth2, s1)

        # Path 1
        depth1_out = self.decode_depth1_step1(s2, s1)

        # 4. Final Concatenation, Inception processing, and Sigmoid probability map
        out = torch.cat((depth4_out, depth3_out, depth2_out, depth1_out), dim=1)
        out = self.inception(out)
        logits = self.final_conv(out)

        return logits

    def init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
            init.kaiming_normal_(m.weight, nonlinearity="relu")


### 1.5. Evaluation functions


In [43]:
def calculate_dice_score(pred_binary, target_binary, smooth=1e-6):
    """
    Calcula la métrica Dice Score (Coeficiente de Dice) para evaluación geométrica.
    Fórmula matemática: DSC = (2 * |A ∩ B| + ε) / (|A| + |B| + ε)
    """
    # Aplanar las dimensiones espaciales, manteniendo la dimensión del lote
    pred = pred_binary.view(pred_binary.size(0), -1)
    target = target_binary.view(target_binary.size(0), -1)

    # |A ∩ B|: Producto elemento a elemento y suma espacial
    intersection = (pred * target).sum(dim=1)

    # |A| + |B|: Suma de áreas individuales
    union = pred.sum(dim=1) + target.sum(dim=1)

    # Cálculo del coeficiente de Dice por cada imagen en el lote
    dice = (2.0 * intersection + smooth) / (union + smooth)

    # Extraer el promedio del lote como un escalar estándar de Python
    return dice.mean().item()


def evaluate_fcn2(model, test_loader):
    """
    Bucle de evaluación del modelo sobre el conjunto de prueba.
    """
    model.eval()

    total_dice = 0.0
    num_batches = 0

    with torch.no_grad():
        for img_batch, mask_batch in test_loader:
            img_batch = img_batch.to(device)
            mask_batch = mask_batch.to(device)

            # 1. Inferencia (Logits)
            logits = model(img_batch)

            # 2. Binarización estricta (Umbralización de probabilidad > 0.5)
            # Esto define la frontera final geométrica para la evaluación
            pred_binary = (torch.sigmoid(logits) > 0.5).float()

            # 3. Evaluación de la métrica
            batch_dice = calculate_dice_score(pred_binary, mask_batch)

            total_dice += batch_dice
            num_batches += 1

    # Promedio total del rendimiento geométrico (DSC ∈ [0, 1])
    avg_dice = total_dice / num_batches

    return avg_dice


### 1.3. FCN1 - Grid Search & Training


### 1.4. Grid Search Execution


In [32]:
def train_model_fcn2(
    model,
    train_loader,
    validation_loader,
    learning_rate=0.001,
    criterion=nn.BCEWithLogitsLoss(),
    epochs=100,
):
    """
    Train the model on the training set and validate it on the validation set.

    Return the training and validation metrics.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    metrics = {
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(epochs):
        losses = 0
        model.train()
        for img, mask in train_loader:
            img = img.to(device).float()
            mask = mask.to(device).float()

            optimizer.zero_grad()
            output = model(img)
            loss = criterion(output, mask)

            loss.backward()
            optimizer.step()
            losses += loss.item()

        metrics["train_loss"].append(losses / len(train_loader))
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for img, mask in validation_loader:
                img = img.to(device)
                mask = mask.to(device)

                output = model(img)
                val_loss += criterion(output, mask).item()

            metrics["val_loss"].append(val_loss / len(validation_loader))

    return metrics

In [33]:
N_EPOCHS = 100


def grid_search():
    batch_sizes = [8, 16, 32]
    lr_values = [0.01, 0.001, 0.0001]

    best_params = {}
    best_metric = 0
    for batch_size in batch_sizes:
        generator = torch.Generator(device="cuda") if device.type == "cuda" else None
        dataloader_args = {"batch_size": batch_size, "generator": generator}

        dl_train_fcn2 = DataLoader(ds_train_fcn2, **dataloader_args, shuffle=True)
        dl_val_fcn2 = DataLoader(ds_validation_fcn2, **dataloader_args, shuffle=False)

        for lr_ in lr_values:
            random.seed(42)
            torch.manual_seed(42)
            np.random.seed(42)

            model = FCN2Net().to(device)
            criterion = nn.BCEWithLogitsLoss()

            print("Training...")
            metrics = train_model_fcn2(
                model,
                dl_train_fcn2,
                dl_val_fcn2,
                criterion=criterion,
                epochs=N_EPOCHS,
                learning_rate=lr_,
            )
            final_train_loss = metrics["train_loss"][-1]
            final_val_loss = metrics["val_loss"][-1]

            print(f"Evaluating model: batch_size={batch_size}, lr_={lr_}, ")
            # 2. Evaluate
            avg_dice = evaluate_fcn2(model, dl_val_fcn2)
            # 3.Compare the results and choose the best hyperparameters
            if avg_dice is not None:
                print(
                    f"batch_size={batch_size}, lr_={lr_}, "
                    f"train_loss={final_train_loss:.5f}, val_loss={final_val_loss:.5f}, "
                    f"avg_dice={avg_dice:.4f} "
                )

                if avg_dice > best_metric:
                    best_metric = avg_dice
                    best_params = {
                        "batch_size": batch_size,
                        "lr_": lr_,
                        "final_train_loss": final_train_loss,
                        "final_val_loss": final_val_loss,
                        "avg_dice": avg_dice,
                    }

    return best_params


best_params = grid_search()
print("\nGrid search finished")
print(f"Best avg_dice = {(best_params.get('avg_dice', -1))}")

print(f"Best batch_size = {best_params.get('batch_size', -1)}")
print(f"Best lr_ = {best_params.get('lr_', -1)}")


Training...
Evaluating model: batch_size=8, lr_=0.01, 
batch_size=8, lr_=0.01, train_loss=0.01657, val_loss=0.02228, avg_dice=0.8951 
Training...
Evaluating model: batch_size=8, lr_=0.001, 
batch_size=8, lr_=0.001, train_loss=0.01133, val_loss=0.02287, avg_dice=0.9021 
Training...
Evaluating model: batch_size=8, lr_=0.0001, 
batch_size=8, lr_=0.0001, train_loss=0.01165, val_loss=0.02462, avg_dice=0.8896 
Training...
Evaluating model: batch_size=16, lr_=0.01, 
batch_size=16, lr_=0.01, train_loss=0.09040, val_loss=0.13192, avg_dice=0.4367 
Training...
Evaluating model: batch_size=16, lr_=0.001, 
batch_size=16, lr_=0.001, train_loss=0.01294, val_loss=0.02154, avg_dice=0.8988 
Training...
Evaluating model: batch_size=16, lr_=0.0001, 
batch_size=16, lr_=0.0001, train_loss=0.01214, val_loss=0.02575, avg_dice=0.8869 
Training...
Evaluating model: batch_size=32, lr_=0.01, 
batch_size=32, lr_=0.01, train_loss=0.01318, val_loss=0.02162, avg_dice=0.9003 
Training...
Evaluating model: batch_size=3

<i>
<h3>Saved Results</h3>
Training...

Evaluating model: batch*size=8, lr*=0.01,

batch*size=8, lr*=0.01, train_loss=0.01657, val_loss=0.02228, avg_dice=0.8951

Training...

Evaluating model: batch*size=8, lr*=0.001,

batch*size=8, lr*=0.001, train_loss=0.01133, val_loss=0.02287, avg_dice=0.9021

Training...

Evaluating model: batch*size=8, lr*=0.0001,

batch*size=8, lr*=0.0001, train_loss=0.01165, val_loss=0.02462, avg_dice=0.8896

Training...

Evaluating model: batch*size=16, lr*=0.01,

batch*size=16, lr*=0.01, train_loss=0.09040, val_loss=0.13192, avg_dice=0.4367

Training...

Evaluating model: batch*size=16, lr*=0.001,

batch*size=16, lr*=0.001, train_loss=0.01294, val_loss=0.02154, avg_dice=0.8988

Training...

Evaluating model: batch*size=16, lr*=0.0001,

batch*size=16, lr*=0.0001, train_loss=0.01214, val_loss=0.02575, avg_dice=0.8869

Training...

Evaluating model: batch*size=32, lr*=0.01,

batch*size=32, lr*=0.01, train_loss=0.01318, val_loss=0.02162, avg_dice=0.9003

Training...

Evaluating model: batch*size=32, lr*=0.001,

batch*size=32, lr*=0.001, train_loss=0.01149, val_loss=0.02298, avg_dice=0.8984

Training...

...

Grid search finished

Best avg_dice = 0.9020947676438552

Best batch_size = 8

Best lr\_ = 0.001

</i>
